<h1 align='center' ><b><i> Crimes In Boston </b></i></h1>

<p align=center>
  <img src="https://live.staticflickr.com/65535/51728157715_ff5aa3f2e3_z.jpg" width="20%">
</p>
This project aims to predict the risk of serious crimes by district in Boston using machine learning.
The goal is to identify which districts are more likely to experience high crime activity in the short term, based on historical and temporal patterns.

The idea behind the project is to support risk prioritization across districts.
Instead of predicting an isolated crime event, the model estimates whether a district is likely to fall into a high-risk pattern in a future time window.

In [28]:
#Import libraries
import numpy as np
import pandas as pd

#Import visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#import sklearn tools
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_recall_curve, 
    classification_report, 
    confusion_matrix, 
    average_precision_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    accuracy_score,
    roc_auc_score,)




In [2]:
#Import the cleaned dataset
df = pd.read_csv('../DataSets/boston_clean.csv', dtype={'INCIDENT_NUMBER': str})

In [3]:
df['Date'] = pd.to_datetime(df['OCCURRED_ON_DATE']).dt.normalize()
df['day'] = df['Date'].dt.floor('D')

In [4]:
#Get only the serious crimes e from Boston districts, excluding the unknown and external ones
df_serious = df.query("DISTRICT != 'Unknown' & DISTRICT != 'External' & UCR_PART == 'Part One'")

The original data was aggregated at the district-day level to transform individual incident records into a temporal series.
This structure made it possible to analyze crime patterns over time and build features based on past behavior.

In [5]:
# Creates a new dataframe with the count of crimes per day for each district
daily = (
    df_serious
    .groupby(['DISTRICT', 'day'])
    .size()
    .reset_index(name='crime_count')
)

In [6]:
# Garantee that the dataframe is sorted by district and day
daily = daily.sort_values(['DISTRICT', 'day'])

To avoid data leakage, all predictive features were built using only information available up to the previous day.
The main features include lagged crime counts, rolling sums over 7, 30, and 90 days, trend indicators, and seasonal variables such as month, day of week, and weekend flags.

In [8]:
# Temporal features based on a 7/30/90-days rolling window for each district
daily['crime_count_lag1'] = daily.groupby('DISTRICT')['crime_count'].shift(1)

daily['crimes_7d'] = (daily).groupby('DISTRICT')['crime_count'].transform(
        lambda s: s.shift(1).rolling(7, min_periods=1).sum()
    )
#     .rolling(7)
#     .sum()
#     .reset_index(level=0, drop=True)
# )

daily['crimes_30d'] = (daily).groupby('DISTRICT')['crime_count'].transform(
        lambda s: s.shift(1).rolling(30, min_periods=1).sum()
    )
#     .rolling(30)
#     .sum()
#     .shift(-30)
#     .reset_index(level=0, drop=True)
# )

daily['crimes_90d'] = (daily).groupby('DISTRICT')['crime_count'].transform(
        lambda s: s.shift(1).rolling(90, min_periods=1).sum()
    )
#     .rolling(90)
#     .sum()
#     .shift(-90)
#     .reset_index(level=0, drop=True)
# )

In [9]:
# Trend feature based on the ratio of crimes in the last 7 days to the last 30 days
daily['trend'] = (
    daily['crimes_7d'] /
    (daily['crimes_30d'] + 1e-6)
)

daily['trend_short_vs_long'] = (
    daily['crimes_7d'] /
    (daily['crimes_90d'] + 1e-6)
)

In [10]:
# Sazional features
daily['month'] = daily['day'].dt.month
daily['dayofweek'] = daily['day'].dt.day_of_week
daily['weekend'] = daily['dayofweek'].isin([5,6]).astype(int)

In [11]:
#Future features: 7 days ahead crime count
daily['target_next_7d'] = (
    daily.groupby('DISTRICT')['crime_count']
    .transform(lambda s: s[::-1].shift(1).rolling(7, min_periods=1).sum()[::-1])
)

In [12]:
#Removes lines without full enough window in the past or in the future
daily = daily.dropna(subset=['crime_count_lag1', 'crimes_7d', 'crimes_30d', 'crimes_90d', 'target_next_7d']).reset_index(drop=True)

The target variable was redefined to represent future risk.
Instead of using the same period as the input features, the model uses the number of serious crimes expected in the next 7 days and converts that value into a binary high-risk label.

In [13]:
#Defines trashold based in the future feateure
threshold = daily['target_next_7d'].quantile(0.75)

daily['high_risk'] = (daily['target_next_7d'] >= threshold).astype(int)

daily.head()

,DISTRICT,day,crime_count,crime_count_lag1,crimes_7d,crimes_30d,crimes_90d,trend,trend_short_vs_long,month,dayofweek,weekend,target_next_7d,high_risk
0,A1,2015-06-16,3,3.0,3.0,3.0,3.0,1.0,1.0,6,1,0,61.0,1
1,A1,2015-06-17,10,3.0,6.0,6.0,6.0,1.0,1.0,6,2,0,57.0,1
2,A1,2015-06-18,7,10.0,16.0,16.0,16.0,1.0,1.0,6,3,0,58.0,1
3,A1,2015-06-19,15,7.0,23.0,23.0,23.0,1.0,1.0,6,4,0,51.0,1
4,A1,2015-06-20,7,15.0,38.0,38.0,38.0,1.0,1.0,6,5,1,52.0,1


The dataset was split chronologically, not randomly.
This is important for time-based problems because it preserves the natural order of events and better simulates a real prediction scenario.

In [14]:
#Order by date to keep the temporal split
daily = daily.sort_values('day').reset_index(drop=True)

In [15]:
#Temporal split
split_date = daily['day'].quantile(0.8)

train = daily[daily['day'] <= split_date].copy()
test = daily[daily['day'] > split_date].copy()

In [16]:
#Diagnosis and protection against single target class

# Mostrar distribuição global do alvo e por partição
print("Target global distribuition (highrisk):")
print(daily['high_risk'].value_counts(dropna=False))
print("\nTrain distribuition:")
print(train['high_risk'].value_counts(dropna=False))
print("\nTest distribuition:")
print(test['high_risk'].value_counts(dropna=False))

#Verify number of classes in train/test
n_classes_train = train['high_risk'].nunique(dropna=True)
n_classes_test = test['high_risk'].nunique(dropna=True)
print(f"\nUnique classes in train: {n_classes_train}, in test: {n_classes_test}")

if n_classes_train < 2:
    print("\nWarning: The training dataset contains only one class. I'm going to try some automatic adjustments to avoid modeling errors.")
    #Reduce treshold to 50% (median) if possible
    new_threshold = daily['target_next_7d'].quantile(0.5)
    print("Trying a treshold based on the median (0.5 quantile) Tentando threshold na mediana (0.5 quantile):", new_threshold)
    daily['highrisk_temp'] = (daily['target_next_7d'] >= new_threshold).astype(int)
    #Recriate time partitions 
    train_temp = daily[daily['day'] <= split_date].copy()
    test_temp = daily[daily['day'] > split_date].copy()
    print("Trainning distribuition with treshold 0.5:")
    print(train_temp['highrisk_temp'].value_counts(dropna=False))

    if train_temp['highrisk_temp'].nunique() >= 2:
        print("OK: median solved. I'm going to use this threshold automatically.")
        daily['highrisk'] = daily['highrisk_temp']
        train = train_temp.copy()
        test = test_temp.copy()
    else:
        #Try 0.4 and 0.3 percentil 
        for q in [0.4, 0.3, 0.2]:
            new_threshold = daily['target_next_7d'].quantile(q)
            daily['highrisk_temp'] = (daily['target_next_7d'] >= new_threshold).astype(int)
            train_temp = daily[daily['day'] <= split_date].copy()
            print(f"Trying treshold {q} quantile : training distrib: {train_temp['highrisk_temp'].value_counts(dropna=False).to_dict()}")
            if train_temp['highrisk_temp'].nunique() >= 2:
                print(f"OK: {q} quantile worked. I'm going to use this treshold automatically({new_threshold}).")
                daily['highrisk'] = daily['highrisk_temp']
                train = train_temp.copy()
                test = daily[daily['day'] > split_date].copy()
                break

    #If nothing works
    if train['highrisk'].nunique() < 2:
        raise RuntimeError(
            "It wasn't possible to get more than one class in the training dataset with automatically threshold adjusts.\n"
            "Sugestions:\n"
            " - reduce the split_date (increase the training data)\n"
            " - use another time features (e.g.: next_14d instead 7d)\n"
            " - accept a treshold much lower manually (e.g.: quantile 0.2) or treat the problem as a regression\n"
            "Ececute one of the options above and generate again 'train' and 'test' before training the model"
        )

else:
    print("\nOK: train contains at least two classes - continue to the normal training.")

# Cleaning: remove the temporary column that was created
if 'highrisk_temp' in daily.columns:
    daily.drop(columns=['highrisk_temp'], inplace=True)

# Mostrar resumo final
print("\nHighrisk summary (after adjustments):")
print(daily['high_risk'].value_counts())
print("\nTrain/test summary (after adjustments):")
print(train['high_risk'].value_counts(), test['high_risk'].value_counts())



Target global distribuition (highrisk):
high_risk
0    26582
1     9474
Name: count, dtype: int64

Train distribuition:
high_risk
0    20792
1     8059
Name: count, dtype: int64

Test distribuition:
high_risk
0    5790
1    1415
Name: count, dtype: int64

Unique classes in train: 2, in test: 2

OK: train contains at least two classes - continue to the normal training.

Highrisk summary (after adjustments):
high_risk
0    26582
1     9474
Name: count, dtype: int64

Train/test summary (after adjustments):
high_risk
0    20792
1     8059
Name: count, dtype: int64 high_risk
0    5790
1    1415
Name: count, dtype: int64


In [17]:
#Features that don't enter as an input
drop_cols = ['high_risk', 'target_next_7d', 'day', 'crime_count']

X_train = train.drop(columns=drop_cols, errors='ignore')
y_train = train['high_risk']

X_test = test.drop(columns=drop_cols, errors='ignore')
y_test = test['high_risk']

#District One-Hot encoding
X_train = pd.get_dummies(X_train, columns=['DISTRICT'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['DISTRICT'], drop_first=False)

#Garantee the same columns in both variables
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

((28851, 21),
 (7205, 21),
 np.float64(0.2793317389345257),
 np.float64(0.1963913948646773))

## Model Comparison Strategy

In this project, two different models will be trained and evaluated in order to identify the most effective one for the task. The goal is not only to compare overall performance, but also to determine which model is better at detecting high-risk districts.

Since this is an imbalanced classification problem, the main metric of interest will be **recall for the positive class**. In this context, recall is the most important measure because it reflects how well the model captures truly high-risk cases and minimizes missed alerts.

Both models will be assessed using the same train-test split, the same evaluation procedure, and the same decision threshold tuning strategy. The final choice will be based on the model that achieves the best recall while maintaining a reasonable balance with precision and overall predictive performance.

In [18]:
num_cols = X_train.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

In [19]:
#Random Forest model
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5],
    'bootstrap': [True],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}

tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=25,
    scoring='average_precision',
    cv=tscv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV score (AP):", search.best_score_)

best_rf = search.best_estimator_
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

Fitting 5 folds for each of 25 candidates, totalling 125 fits


[CV] END bootstrap=True, class_weight=None, max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   6.1s
[CV] END bootstrap=True, class_weight=None, max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   8.5s
[CV] END bootstrap=True, class_weight=None, max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=  11.8s
[CV] END bootstrap=True, class_weight=None, max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=  11.3s
[CV] END bootstrap=True, class_weight=None, max_depth=20, max_features=log2, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time=   8.4s
[CV] END bootstrap=True, class_weight=balanced, max_depth=5, max_features=0.5, min_samples_leaf=1, min_samples_split=20, n_estimators=500; total time=   7.1s
[CV] END bootstrap=True, class_weight=balanced, max_depth=5, m

In [23]:
#Random Forest Evaluation
def evaluate_model(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]

    y_pred = (y_proba >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "threshold": best_threshold,
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_1": precision_score(y_true, y_pred, zero_division=0),
        "recall_1": recall_score(y_true, y_pred, zero_division=0),
        "f1_1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

precision, recall, thresholds = precision_recall_curve(y_test, y_proba_rf)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_thr_rf = thresholds[best_idx]

y_pred_rf = (y_proba_rf >= best_thr_rf).astype(int)

print("Best threshold:", best_thr_rf)
print("PR AUC:", average_precision_score(y_test, y_proba_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=4, zero_division=0))

Best threshold: 0.519197487625799
PR AUC: 0.8671393961278532
[[5404  386]
 [ 277 1138]]
              precision    recall  f1-score   support

           0     0.9512    0.9333    0.9422      5790
           1     0.7467    0.8042    0.7744      1415

    accuracy                         0.9080      7205
   macro avg     0.8490    0.8688    0.8583      7205
weighted avg     0.9111    0.9080    0.9092      7205



Because the dataset is highly imbalanced, the logistic regression model was trained with balanced class weights.
This helps the model pay more attention to the rare positive class instead of ignoring it.<br>
Also standard accuracy was not enough to evaluate the model properly.
Since the positive class is rare, the project focuses on ROC AUC, PR AUC, precision, recall, F1-score, and the confusion matrix.

In [26]:
#Pipeline to train the model
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=2000, 
        random_state=42,
        class_weight='balanced'))
])

model.fit(X_train, y_train)

y_pred_lr = model.predict(X_test)
y_proba_lr = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_lr)

# f1 for each treshold
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("Precision:", precisions[best_idx])
print("Recall:", recalls[best_idx])
print("F1:", f1_scores[best_idx])

y_pred_best = (y_proba_lr >= best_threshold).astype(int)
print(confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best, digits=4, zero_division=0))

Best threshold: 0.5699154524349126
Precision: 0.7154761904761905
Recall: 0.8494699646643109
F1: 0.7767366715553619
[[5312  478]
 [ 213 1202]]
              precision    recall  f1-score   support

           0     0.9614    0.9174    0.9389      5790
           1     0.7155    0.8495    0.7767      1415

    accuracy                         0.9041      7205
   macro avg     0.8385    0.8835    0.8578      7205
weighted avg     0.9131    0.9041    0.9071      7205



The default decision threshold of 0.50 was not ideal for this problem.
To improve the balance between precision and recall, the threshold was tuned using the precision-recall curve, which led to a better classification strategy for the rare class.

In [29]:


def best_threshold_from_pr(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
    best_idx = np.argmax(f1_scores)
    return {
        'precision_curve': precision,
        'recall_curve': recall,
        'thresholds': thresholds,
        'best_idx': best_idx,
        'best_threshold': thresholds[best_idx],
        'best_precision': precision[best_idx],
        'best_recall': recall[best_idx],
        'best_f1': f1_scores[best_idx],
        'pr_auc': average_precision_score(y_true, y_proba),
        'roc_auc': roc_auc_score(y_true, y_proba)
    }

def eval_model(y_true, y_proba):
    out = best_threshold_from_pr(y_true, y_proba)
    y_pred = (y_proba >= out['best_threshold']).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    out.update({
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_1': precision_score(y_true, y_pred, zero_division=0),
        'recall_1': recall_score(y_true, y_pred, zero_division=0),
        'f1_1': f1_score(y_true, y_pred, zero_division=0),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'y_pred': y_pred
    })
    return out

lr_eval = eval_model(y_test, y_proba_lr)
rf_eval = eval_model(y_test, y_proba_rf)

comparison_df = pd.DataFrame([
    {
        'model': 'Logistic Regression',
        'threshold': lr_eval['best_threshold'],
        'roc_auc': lr_eval['roc_auc'],
        'pr_auc': lr_eval['pr_auc'],
        'accuracy': lr_eval['accuracy'],
        'precision_1': lr_eval['precision_1'],
        'recall_1': lr_eval['recall_1'],
        'f1_1': lr_eval['f1_1'],
        'tn': lr_eval['tn'],
        'fp': lr_eval['fp'],
        'fn': lr_eval['fn'],
        'tp': lr_eval['tp']
    },
    {
        'model': 'Random Forest',
        'threshold': rf_eval['best_threshold'],
        'roc_auc': rf_eval['roc_auc'],
        'pr_auc': rf_eval['pr_auc'],
        'accuracy': rf_eval['accuracy'],
        'precision_1': rf_eval['precision_1'],
        'recall_1': rf_eval['recall_1'],
        'f1_1': rf_eval['f1_1'],
        'tn': rf_eval['tn'],
        'fp': rf_eval['fp'],
        'fn': rf_eval['fn'],
        'tp': rf_eval['tp']
    }
]).round(4)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f"Precision-Recall Curve - Logistic Regression (AP={lr_eval['pr_auc']:.3f})",
        f"Precision-Recall Curve - Random Forest (AP={rf_eval['pr_auc']:.3f})",
        "Confusion Matrix - Logistic Regression",
        "Confusion Matrix - Random Forest"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# PR curves
fig.add_trace(
    go.Scatter(
        x=lr_eval['recall_curve'],
        y=lr_eval['precision_curve'],
        mode='lines',
        name='LR PR curve',
        line=dict(color='#1f77b4', width=3)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=rf_eval['recall_curve'],
        y=rf_eval['precision_curve'],
        mode='lines',
        name='RF PR curve',
        line=dict(color='#d62728', width=3)
    ),
    row=1, col=2
)

# Threshold points
fig.add_trace(
    go.Scatter(
        x=[lr_eval['best_recall']],
        y=[lr_eval['best_precision']],
        mode='markers+text',
        marker=dict(size=10, color='black'),
        text=[f"{lr_eval['best_threshold']:.3f}"],
        textposition='top center',
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=[rf_eval['best_recall']],
        y=[rf_eval['best_precision']],
        mode='markers+text',
        marker=dict(size=10, color='black'),
        text=[f"{rf_eval['best_threshold']:.3f}"],
        textposition='top center',
        showlegend=False
    ),
    row=1, col=2
)

# Confusion matrices
cm_lr = confusion_matrix(y_test, lr_eval['y_pred'])
cm_rf = confusion_matrix(y_test, rf_eval['y_pred'])

fig.add_trace(
    go.Heatmap(
        z=cm_lr,
        x=['Pred 0', 'Pred 1'],
        y=['True 0', 'True 1'],
        colorscale='Blues',
        text=cm_lr,
        texttemplate='%{text}',
        showscale=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Heatmap(
        z=cm_rf,
        x=['Pred 0', 'Pred 1'],
        y=['True 0', 'True 1'],
        colorscale='Blues',
        text=cm_rf,
        texttemplate='%{text}',
        showscale=True
    ),
    row=2, col=2
)

fig.update_layout(
    title='Final Model Comparison',
    template='plotly_white',
    width=1400,
    height=1000
)

display(comparison_df)
fig.show()

,model,threshold,roc_auc,pr_auc,accuracy,precision_1,recall_1,f1_1,tn,fp,fn,tp
0,Logistic Regression,0.5699,0.9588,0.8621,0.9041,0.7155,0.8495,0.7767,5312,478,213,1202
1,Random Forest,0.5192,0.9599,0.8671,0.9080,0.7467,0.8042,0.7744,5404,386,277,1138


## Final Model Selection

Both Logistic Regression and Random Forest performed competitively after threshold tuning. Logistic Regression delivered the highest recall for the positive class, while Random Forest achieved slightly better precision and ranking metrics.

Since the project is focused on early identification of high-risk districts, recall is the most relevant metric for model selection. Missing a truly high-risk district would be more costly than flagging an additional false alarm, so the final model should be chosen based on its ability to maximize recall for the positive class.

## Final Remarks

This project aimed to predict the risk of serious crimes across Boston districts using a time-based machine learning approach. After restructuring the problem as a future risk prediction task, the analysis focused on building a pipeline that avoided temporal leakage, captured historical crime dynamics, and evaluated model performance under class imbalance.

Two models were tested: Logistic Regression and Random Forest. After applying the same temporal split and threshold-tuning procedure to both models, their performance turned out to be very competitive. Logistic Regression achieved slightly higher recall for the positive class, while Random Forest showed marginally better results in precision, ROC AUC, PR AUC, and overall accuracy.

Since the main objective of the project is to identify high-risk districts and minimize missed alerts, recall was treated as the most important metric for model selection. In this context, the preferred model is the one that captures a larger share of truly high-risk districts, even if that comes with a different balance of false positives.

Overall, this project demonstrates how temporal feature engineering, chronological validation, threshold calibration, and metric selection can significantly improve the quality of a predictive pipeline for imbalanced classification problems. Beyond the final model itself, the main contribution of this work is the development of a more realistic and methodologically sound framework for district-level crime risk prediction.